# Prometheus - Medical Image Segmentation Training on PUMA

**Dataset:** [PUMA](https://puma.grand-challenge.org/dataset/) — Panoptic segmentation of nUclei and tissue in MelanomA

**Models:** UNetTissue (6 tissue classes) | DualUNet (6 tissue + 10 nuclei classes)

**GPU:** Google Colab G4 / A100 100GB VRAM

---
## 1. Setup

In [ ]:
import os, sys

# Clone repo and install package (Colab setup)
REPO_DIR = "/content/prometheus"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hoangtung386/Prometheus.git {REPO_DIR}

In [ ]:
%cd {REPO_DIR}

In [ ]:
!pip install -qq -e .
!pip install -qq tensorboard matplotlib

In [ ]:
# Clear cached prometheus modules and add src to path
for mod in list(sys.modules.keys()):
    if mod.startswith('prometheus'):
        del sys.modules[mod]
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

import torch
import torch.optim as optim
from torch.utils.data import DataLoader

print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}")
for i in range(torch.cuda.device_count()):
    print(f"  [{i}] {torch.cuda.get_device_name(i)} "
          f"({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from prometheus import (
    UNetTissue,
    DualUNet,
    CombinedLoss,
    Trainer,
    visualize_sample,
    show_prediction,
    predict_sample,
)
from prometheus.config import ModelConfig, TrainingConfig, DEFAULT_CONFIG, DEFAULT_TRAINING_CONFIG
from prometheus.data import (
    PUMADataset,
    create_puma_dataloaders,
    train_transform,
    val_transform,
    collate_puma,
)
from prometheus.data.puma_dataset import (
    geojson_to_mask,
    TISSUE_CLASSES,
    NUCLEI_CLASSES,
    TISSUE_CLASS_TO_IDX,
    NUCLEI_CLASS_TO_IDX,
)
from prometheus.data.transforms import (
    Compose,
    RandomHorizontalFlip,
    RandomVerticalFlip,
    RandomRotate90,
    ElasticDeformation,
    NormalizeTile,
    RandomBrightnessContrast,
    RandomGaussianNoise,
    Normalize,
)
from prometheus.models import (
    Encoder,
    Decoder,
    TissueAttentionEncoder,
    TissueDecoder,
    build_encoder,
    forward_encoder,
    build_decoder,
    forward_decoder,
)
from prometheus.training import (
    dice_score,
    warmup_cosine_lr,
    train_one_epoch,
    validate,
)
from prometheus.losses import (
    BCEWithLogitsLoss,
    DiceLoss,
    FocalLoss,
    MultiClassDiceLoss,
    TverskyLoss,
)
from prometheus.blocks import (
    ConvNeXtBlock,
    DecoderBlock,
    LocalGlobalAttention,
    SparseMoE,
    Expert,
    EncoderTransformerBlock,
    EncoderTransformerStack,
)
from prometheus.utils import LayerNorm, GRN

print("All imports from prometheus codebase successful!")

---
## 2. Configuration

Sử dụng `DEFAULT_CONFIG` và `DEFAULT_TRAINING_CONFIG` làm baseline, sau đó customize cho PUMA dataset.

In [ ]:
print("Default Model Config:")
print(f"  encoder_dims: {DEFAULT_CONFIG.encoder_dims}")
print(f"  encoder_depths: {DEFAULT_CONFIG.encoder_depths}")
print(f"  drop_path_rate: {DEFAULT_CONFIG.drop_path_rate}")
print(f"  num_transformer_blocks: {DEFAULT_CONFIG.num_transformer_blocks}")
print()
print("Default Training Config:")
print(f"  batch_size: {DEFAULT_TRAINING_CONFIG.batch_size}")
print(f"  epochs: {DEFAULT_TRAINING_CONFIG.epochs}")
print(f"  lr: {DEFAULT_TRAINING_CONFIG.lr}")
print(f"  warmup_epochs: {DEFAULT_TRAINING_CONFIG.warmup_epochs}")

In [ ]:
model_cfg = ModelConfig(
    in_chans=3,
    num_classes=11,
    drop_path_rate=0.1,
)

train_cfg = TrainingConfig(
    model_type="DualUNet",
    image_size=1024,
    num_tissue_classes=6,
    num_nuclei_classes=10,
    batch_size=16,
    epochs=200,
    lr=2e-4,
    weight_decay=1e-2,
    warmup_epochs=10,
    gradient_clip_norm=1.0,
    amp=True,
    data_root="/content/drive/MyDrive/puma_data",
    num_workers=8,
    val_split=0.1,
    log_dir="/content/drive/MyDrive/prometheus_logs",
    ckpt_dir="/content/drive/MyDrive/prometheus_checkpoints",
    log_interval=10,
    save_interval=5,
    moe_loss_weight=0.01,
)

os.makedirs(train_cfg.log_dir, exist_ok=True)
os.makedirs(train_cfg.ckpt_dir, exist_ok=True)

print(f"\nCustomized Config:")
print(f"  Model: {train_cfg.model_type}")
print(f"  Tissue classes: {train_cfg.num_tissue_classes}, Nuclei classes: {train_cfg.num_nuclei_classes}")
print(f"  Image size: {train_cfg.image_size}x{train_cfg.image_size}, Batch: {train_cfg.batch_size}")

---
## 3. Dataset — PUMA (tissue 6 classes + nuclei 10 classes)

Sử dụng `train_transform` và `val_transform` cho augmentation pipeline.

In [ ]:
print("PUMA Dataset Classes:")
print(f"\nTissue ({len(TISSUE_CLASSES)} classes):")
for name, idx in TISSUE_CLASS_TO_IDX.items():
    print(f"  {idx}: {name}")

print(f"\nNuclei ({len(NUCLEI_CLASSES)} classes):")
for name, idx in NUCLEI_CLASS_TO_IDX.items():
    print(f"  {idx}: {name}")

In [ ]:
print("Augmentation Pipeline:")
print(f"\nTrain transforms:")
train_aug = train_transform()
for t in train_aug.transforms:
    print(f"  - {t.__class__.__name__}")

print(f"\nVal transforms:")
val_aug = val_transform()
for t in val_aug.transforms:
    print(f"  - {t.__class__.__name__}")

In [ ]:
if os.path.exists(train_cfg.data_root):
    train_loader, val_loader = create_puma_dataloaders(
        root=train_cfg.data_root,
        image_size=train_cfg.image_size,
        batch_size=train_cfg.batch_size,
        num_workers=train_cfg.num_workers,
        val_split=train_cfg.val_split,
        seed=train_cfg.seed,
        train_transforms=train_transform(),
        val_transforms=val_transform(),
    )
    print(f"Train and val dataloaders created with transforms pipeline")
else:
    print(f"Data not found at {train_cfg.data_root}")
    print("Falling back to dummy data for testing...")
    train_loader = val_loader = None

### 3a. Quick visualization

In [ ]:
if os.path.exists(train_cfg.data_root):
    ds = PUMADataset(
        root=train_cfg.data_root,
        image_size=train_cfg.image_size,
        augment=False,
        transforms=val_transform(),
    )
    visualize_sample(ds, 0)

### 3b. Custom transform example

Tạo custom augmentation pipeline với các transform components từ codebase.

In [ ]:
custom_aug = Compose([
    RandomHorizontalFlip(p=0.5),
    RandomVerticalFlip(p=0.5),
    RandomRotate90(),
    ElasticDeformation(alpha=25, sigma=4, p=0.3),
    RandomBrightnessContrast(brightness=0.05, contrast=0.1),
    RandomGaussianNoise(std=0.01),
    NormalizeTile(),
])

print("Custom augmentation pipeline:")
for t in custom_aug.transforms:
    print(f"  - {t.__class__.__name__}")

if os.path.exists(train_cfg.data_root):
    ds_custom = PUMADataset(
        root=train_cfg.data_root,
        image_size=train_cfg.image_size,
        augment=True,
        transforms=custom_aug,
    )
    print(f"\nDataset with custom transforms: {len(ds_custom)} samples")
    visualize_sample(ds_custom, 0)

### 3c. Verify data pipeline

Kiểm tra augment đồng bộ image+mask, normalization stats, và shuffle trên tập train.

In [ ]:
import matplotlib.pyplot as plt

if os.path.exists(train_cfg.data_root):
    ds_raw = PUMADataset(
        root=train_cfg.data_root,
        image_size=train_cfg.image_size,
        augment=False,
        transforms=None,
    )
    ds_aug = PUMADataset(
        root=train_cfg.data_root,
        image_size=train_cfg.image_size,
        augment=True,
        transforms=None,
    )

    idx = 0
    img_raw, tgt_raw = ds_raw[idx]
    img_aug, tgt_aug = ds_aug[idx]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    axes[0, 0].imshow(img_raw.permute(1, 2, 0).clip(0, 1) if img_raw.max() <= 1 else img_raw.permute(1, 2, 0) / 255)
    axes[0, 0].set_title("Original image")
    axes[0, 1].imshow(tgt_raw["tissue"].argmax(dim=0), cmap="tab10", vmin=0, vmax=5)
    axes[0, 1].set_title("Original tissue mask")
    axes[0, 2].imshow(tgt_raw["nuclei"].argmax(dim=0), cmap="tab10", vmin=0, vmax=10)
    axes[0, 2].set_title("Original nuclei mask")

    axes[1, 0].imshow(img_aug.permute(1, 2, 0).clip(0, 1) if img_aug.max() <= 1 else img_aug.permute(1, 2, 0) / 255)
    axes[1, 0].set_title("Augmented image")
    axes[1, 1].imshow(tgt_aug["tissue"].argmax(dim=0), cmap="tab10", vmin=0, vmax=5)
    axes[1, 1].set_title("Augmented tissue mask")
    axes[1, 2].imshow(tgt_aug["nuclei"].argmax(dim=0), cmap="tab10", vmin=0, vmax=10)
    axes[1, 2].set_title("Augmented nuclei mask")

    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    print("Augment uses np.flip(axis=2) for horizontal, np.flip(axis=1) for vertical, np.rot90(axes=(1,2)) for rotation")
    print("All spatial transforms apply identically to image AND masks")

In [ ]:
if os.path.exists(train_cfg.data_root):
    ds_no_norm = PUMADataset(
        root=train_cfg.data_root,
        image_size=train_cfg.image_size,
        augment=False,
        transforms=None,
    )
    ds_norm = PUMADataset(
        root=train_cfg.data_root,
        image_size=train_cfg.image_size,
        augment=False,
        transforms=val_transform(),
    )

    img_before, _ = ds_no_norm[0]
    img_after, _ = ds_norm[0]

    print("Normalization verification:")
    print(f"  Before: mean={img_before.mean():.4f}, std={img_before.std():.4f}, "
          f"min={img_before.min():.4f}, max={img_before.max():.4f}")
    print(f"  After:  mean={img_after.mean():.4f}, std={img_after.std():.4f}, "
          f"min={img_after.min():.4f}, max={img_after.max():.4f}")
    print()
    print("Per-channel stats (after NormalizeTile):")
    for c in range(img_after.shape[0]):
        ch = img_after[c]
        print(f"  Channel {c}: mean={ch.mean():.4f}, std={ch.std():.4f}, "
              f"min={ch.min():.4f}, max={ch.max():.4f}")

In [ ]:
print("Shuffle verification on train DataLoader:")
print(f"  train_loader.shuffle = {train_loader is not None}")

if train_loader is not None:
    first_batch_1 = next(iter(train_loader))
    first_batch_2 = next(iter(train_loader))

    same = torch.equal(first_batch_1[0][:1], first_batch_2[0][:1])
    print(f"  Two consecutive iterations return same first sample: {same}")
    print(f"  → Shuffle is {'OFF (unexpected!)' if same else 'ON (correct)'}")
    print(f"  Batch shape: images={first_batch_1[0].shape}, "
          f"tissue={first_batch_1[1]['tissue'].shape}, "
          f"nuclei={first_batch_1[1]['nuclei'].shape}")

---
## 4. Model Architecture

Inspect model components: `Encoder`, `Decoder`, `TissueAttentionEncoder`, `TissueDecoder`, và building blocks.

In [ ]:
if train_cfg.model_type == "UNetTissue":
    model_cfg = ModelConfig(
        in_chans=3,
        num_classes=train_cfg.num_tissue_classes,
        drop_path_rate=0.1,
    )
    model = UNetTissue(config=model_cfg)
else:
    model_cfg = ModelConfig(
        in_chans=3,
        num_tissue_classes=train_cfg.num_tissue_classes,
        num_nuclei_classes=train_cfg.num_nuclei_classes + 1,
        drop_path_rate=0.1,
    )
    model = DualUNet(config=model_cfg)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{train_cfg.model_type}: {total:,} params ({trainable:,} trainable)")

### 4a. Inspect model components

In [ ]:
print("Model Architecture Components:")
print(f"\n{train_cfg.model_type} structure:")

if train_cfg.model_type == "UNetTissue":
    print(f"  Encoder: {model.encoder.__class__.__name__}")
    print(f"    - stem: {model.encoder.stem}")
    print(f"    - stages: {len(model.encoder.stages)} stages")
    print(f"  Decoder: {model.decoder.__class__.__name__}")
    print(f"    - levels: {len(model.decoder.levels)} levels")
else:
    print(f"  Tissue stream:")
    print(f"    - tissue_stem: {model.tissue_stem}")
    print(f"    - tissue_decoder: {model.tissue_decoder.__class__.__name__}")
    print(f"  Tissue→Nuclei bridge:")
    print(f"    - tissue_attention_encoder: {model.tissue_attention_encoder.__class__.__name__}")
    print(f"  Nuclei stream:")
    print(f"    - nuclei_stem: {model.nuclei_stem}")
    print(f"    - transformer: {model.transformer.__class__.__name__}")
    print(f"      - {len(model.transformer.blocks)} transformer blocks")

### 4b. Building blocks inspection

In [ ]:
print("Building Blocks from codebase:")

print("\n1. ConvNeXtBlock (encoder backbone):")
convnext = ConvNeXtBlock(dim=96, drop_path=0.1)
print(f"   Parameters: {sum(p.numel() for p in convnext.parameters()):,}")

print("\n2. DecoderBlock (decoder with skip connection):")
decoder_blk = DecoderBlock(dim=192, has_upsample=True, in_dim=96)
print(f"   Parameters: {sum(p.numel() for p in decoder_blk.parameters()):,}")

print("\n3. LocalGlobalAttention (transformer attention):")
lga = LocalGlobalAttention(d_model=768, n_heads=8, window_size=4)
print(f"   Parameters: {sum(p.numel() for p in lga.parameters()):,}")

print("\n4. SparseMoE (mixture of experts):")
moe = SparseMoE(d_model=768, d_expert=256, num_experts=8, top_k=2)
print(f"   Parameters: {sum(p.numel() for p in moe.parameters()):,}")

print("\n5. EncoderTransformerBlock (full transformer block):")
trans_blk = EncoderTransformerBlock(
    d_model=768, n_heads=8, d_ff=3072,
    d_expert=256, num_experts=8, top_k=2
)
print(f"   Parameters: {sum(p.numel() for p in trans_blk.parameters()):,}")

print("\n6. LayerNorm & GRN (normalization):")
ln = LayerNorm(96, eps=1e-6, data_format="channels_first")
grn = GRN(384)
print(f"   LayerNorm params: {sum(p.numel() for p in ln.parameters()):,}")
print(f"   GRN params: {sum(p.numel() for p in grn.parameters()):,}")

### 4c. Manual encoder/decoder construction

In [ ]:
print("Manual encoder/decoder construction using build_encoder/build_decoder:")

stem, downsample_layers, stages = build_encoder(
    in_chans=3,
    dims=[96, 192, 384, 768],
    depths=[3, 3, 9, 3],
    drop_path_rate=0.1,
)

print(f"\nEncoder components:")
print(f"  stem: {stem}")
print(f"  downsample_layers: {len(downsample_layers)} layers")
print(f"  stages: {len(stages)} stages")
for i, stage in enumerate(stages):
    print(f"    stage {i}: {len(stage)} blocks")

levels, output_head = build_decoder(
    encoder_dims=[96, 192, 384, 768],
    encoder_depths=[3, 3, 9, 3],
    num_classes=11,
)

print(f"\nDecoder components:")
print(f"  levels: {len(levels)} levels")
for i, level in enumerate(levels):
    print(f"    level {i}: {len(level)} blocks")
print(f"  output_head: {output_head}")

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
print("Cleared CPU RAM and GPU VRAM")

---
## 5. Loss Functions

So sánh các loss function từ codebase.

In [ ]:
print("Available Loss Functions:")

dummy_logits = torch.randn(2, 6, 64, 64)
dummy_targets = torch.randint(0, 2, (2, 6, 64, 64)).float()

losses = {
    "BCEWithLogitsLoss": BCEWithLogitsLoss(),
    "DiceLoss": DiceLoss(),
    "FocalLoss": FocalLoss(alpha=0.25, gamma=2.0),
    "TverskyLoss": TverskyLoss(alpha=0.3, beta=0.7),
    "CombinedLoss (BCE+Dice)": CombinedLoss(bce_weight=1.0, dice_weight=1.0),
}

print("\nLoss values on dummy data:")
for name, loss_fn in losses.items():
    loss_val = loss_fn(dummy_logits, dummy_targets)
    print(f"  {name}: {loss_val.item():.4f}")

In [ ]:
gc.collect()
torch.cuda.empty_cache()
print("Cleared CPU RAM and GPU VRAM before training")

---
## 6. Training

### 6a. Manual training loop

Sử dụng `train_one_epoch`, `validate`, `warmup_cosine_lr`, `dice_score` cho fine-grained control.

In [ ]:
if train_loader is None:
    print("=== DUMMY DATA MODE ===")
    dummy = [(torch.randn(3, 256, 256),
              {"tissue": torch.randint(0, 2, (6, 256, 256)).float(),
               "nuclei": torch.randint(0, 2, (11, 256, 256)).float()})
             for _ in range(64)]
    train_loader = val_loader = DataLoader(dummy, batch_size=4, shuffle=True, collate_fn=collate_puma)
    train_cfg.epochs = 5

model.to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=train_cfg.lr,
    weight_decay=train_cfg.weight_decay,
    betas=train_cfg.betas,
    eps=train_cfg.eps,
)

scheduler = optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda epoch: warmup_cosine_lr(epoch, train_cfg.warmup_epochs, train_cfg.epochs)
)

scaler = torch.cuda.amp.GradScaler(enabled=train_cfg.amp)
criterion = CombinedLoss(bce_weight=1.0, dice_weight=1.0)

print("Manual training setup complete!")
print(f"  Optimizer: AdamW (lr={train_cfg.lr})")
print(f"  Scheduler: warmup_cosine_lr")
print(f"  Loss: CombinedLoss (BCE + Dice)")
print(f"  AMP: {train_cfg.amp}")

In [ ]:
print("Running 2 manual epochs with train_one_epoch and validate:")

for epoch in range(2):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, epoch, train_cfg, device
    )
    scheduler.step()
    
    val_loss, val_dice_t, val_dice_n = validate(
        model, val_loader, criterion, train_cfg, device
    )
    
    lr_now = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}, "
          f"dice_t={val_dice_t:.4f}, dice_n={val_dice_n:.4f}, lr={lr_now:.2e}")

### 6b. Using Trainer for full training pipeline

In [ ]:
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_cfg,
    device=device,
)

best_dice = trainer.fit()
print(f"Training completed! Best dice: {best_dice:.4f}")

In [ ]:
gc.collect()
torch.cuda.empty_cache()
print("Cleared CPU RAM and GPU VRAM before evaluation")

---
## 7. Evaluation & Inference

Sử dụng `dice_score`, `predict_sample`, `show_prediction` cho evaluation.

In [ ]:
print("Evaluation using dice_score:")

model.eval()
with torch.no_grad():
    for images, targets in val_loader:
        images = images.to(device)
        t_mask = targets["tissue"].to(device)
        n_mask = targets["nuclei"].to(device)
        
        if train_cfg.model_type == "UNetTissue":
            logits = model(images)
            dice_t = dice_score(logits, t_mask)
            print(f"  Tissue Dice: {dice_t.mean().item():.4f}")
        else:
            pred_t, pred_n, _ = model(images)
            dice_t = dice_score(pred_t, t_mask)
            dice_n = dice_score(pred_n, n_mask)
            print(f"  Tissue Dice: {dice_t.mean().item():.4f}")
            print(f"  Nuclei Dice: {dice_n.mean().item():.4f}")
        
        break

In [ ]:
if os.path.exists(train_cfg.data_root):
    ds = PUMADataset(
        root=train_cfg.data_root,
        image_size=train_cfg.image_size,
        augment=False,
        transforms=val_transform(),
    )
    
    print("Inference using predict_sample:")
    img, targets = ds[0]
    pred = predict_sample(model, img, train_cfg.model_type, device)
    
    if train_cfg.model_type == "UNetTissue":
        print(f"  Prediction shape: {pred.shape}")
        print(f"  Unique classes: {torch.unique(torch.from_numpy(pred))}")
    else:
        print(f"  Tissue prediction shape: {pred['tissue'].shape}")
        print(f"  Nuclei prediction shape: {pred['nuclei'].shape}")
        print(f"  Tissue unique classes: {torch.unique(torch.from_numpy(pred['tissue']))}")
        print(f"  Nuclei unique classes: {torch.unique(torch.from_numpy(pred['nuclei']))}")
    
    print("\nVisualization using show_prediction:")
    show_prediction(model, ds, 0, train_cfg.model_type, device)

---
## 8. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {train_cfg.log_dir}